# Preparation using huggingface models

In [ ]:
!pip install -U bitsandbytes>=0.46.1

In [ ]:
!pip install -q langchain langchain_community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 27.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 4.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.33.1 which is incompatible.


In [ ]:
!pip install -q pypdf transformers sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 333.7/333.7 kB 5.1 MB/s eta 0:00:00


In [ ]:
!pip install -q faiss-gpu accelerate rerankers

ERROR: Could not find a version that satisfies the requirement faiss-gpu (from versions: none)
ERROR: No matching distribution found for faiss-gpu


## RETRIEVER - LLM

### Raw Knowledge Base preparation

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from tqdm import tqdm
import os

# Update these paths to where you uploaded the files in your Google Drive
# Example: If you uploaded them to a folder named 'my_docs'
DRIVE_PATH = "/content/drive/MyDrive/"

PDF_LIST = [
    os.path.join(DRIVE_PATH, "crime_act.pdf"),
    os.path.join(DRIVE_PATH, "interim_government_act.pdf"),
    os.path.join(DRIVE_PATH, "labor_act.pdf")
]

RAW_KNOWLEDGE_BASE = []

for pdf_path in tqdm(PDF_LIST, desc="Loading PDFs"):
    if os.path.exists(pdf_path):
        loader = PyPDFLoader(pdf_path)
        pages = loader.load()
        RAW_KNOWLEDGE_BASE.extend(pages)
    else:
        print(f"File not found: {pdf_path}")

print(f"\nTotal pages loaded: {len(RAW_KNOWLEDGE_BASE)}")

Loading PDFs: 100%|██████████| 3/3 [00:10<00:00,  3.53s/it]


Total pages loaded: 141


### Document preparation

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from transformers import AutoTokenizer
from typing import List, Optional
from langchain_core.documents import Document
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy
import torch

# Check if CUDA is available for the embedding model
device = "cuda" if torch.cuda.is_available() else "cpu"

EMBEDDING_MODEL_NAME = "thenlper/gte-small"

MARKDOWN_SEPARATORS = [
    "\n#{1,6} ",
    "```\n",
    "\n\\*\\*\\*+\n",
    "\n---+\n",
    "\n___+\n",
    "\n\n",
    "\n",
    " ",
    "",
]

def split_documents(
    chunk_size: int,
    knowledge_base: List[Document],
    tokenizer_name: Optional[str] = EMBEDDING_MODEL_NAME,
) -> List[Document]:

    # In Colab, we ensure the tokenizer is downloaded to the local cache
    tokenizer = AutoTokenizer.from_pretrained(tokenizer_name)

    text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
        tokenizer,
        chunk_size=chunk_size,
        chunk_overlap=int(chunk_size / 10),
        add_start_index=True,
        strip_whitespace=True,
        separators=MARKDOWN_SEPARATORS,
    )

    docs_processed = []
    # Using tqdm for progress tracking in Colab cells
    from tqdm.notebook import tqdm
    for doc in tqdm(knowledge_base, desc="Splitting documents"):
        docs_processed += text_splitter.split_documents([doc])

    unique_texts = {}
    docs_processed_unique = []
    for doc in docs_processed:
        if doc.page_content not in unique_texts:
            unique_texts[doc.page_content] = True
            docs_processed_unique.append(doc)

    return docs_processed_unique

### The setup

In [ ]:
docs_processed = split_documents(
    512,
    RAW_KNOWLEDGE_BASE,
    tokenizer_name=EMBEDDING_MODEL_NAME,
)

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/394 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

Splitting documents:   0%|          | 0/141 [00:00<?, ?it/s]

### Building the vector database

In [ ]:
!pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 48.6 MB/s eta 0:00:00


In [ ]:
embedding_model = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL_NAME,
    multi_process=False,
    model_kwargs={"device": device},
    encode_kwargs={"normalize_embeddings": True},
)

# Build the Vector Database
KNOWLEDGE_VECTOR_DATABASE = FAISS.from_documents(
    docs_processed, embedding_model, distance_strategy=DistanceStrategy.COSINE
)

print(f"Vector database created with {KNOWLEDGE_VECTOR_DATABASE.index.ntotal} entries.")

/tmp/ipykernel_3096/2348235652.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(


modules.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/66.7M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: thenlper/gte-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector database created with 153 entries.


In [ ]:
user_query = "What is the meaning of 'Basic remuneration'?"
query_vector = embedding_model.embed_query(user_query)

In [ ]:
print(f"\nStarting retrieval for {user_query=}...")
retrieved_docs = KNOWLEDGE_VECTOR_DATABASE.similarity_search(query=user_query, k=5)
print(
    "\n==================================Top document=================================="
)
print(retrieved_docs[0].page_content)
print("==================================Metadata==================================")
print(retrieved_docs[0].metadata)


Starting retrieval for user_query="What is the meaning of 'Basic remuneration'?"...

==================================Top document==================================
www.lawcommission.gov.np 
14 
 
Chapter-8 
Provisions Relating to Remuneration 
34. Entitlement of labours to remuneration: (1) Each worker shall be entitled to 
receive the remuneration and benefits from the date  on which he or she starts the 
work.  
(2) The remuneration and benefits which a labour is entitled to shall be 
so specified in the employment contract as not to be less t han that specified in th is 
Act and the rules framed under this Act.  
(3) Except as mentioned in the collective agreement between the 
employer and the labour, the remuneration and benefi ts being received and 
enjoyed by the labour shall not be decreased.  
35. Payment of remuneration:  (1) In paying the remuneration to the labour, the 
employer shall make such payment in accordance with t he provision, if any, 
mentioned in the employmen

## READER - LLM

In [ ]:
!pip install rerankers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 3.3 MB/s eta 0:00:00


### Prompt

In [ ]:
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

READER_MODEL_NAME = "HuggingFaceH4/zephyr-7b-beta"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
model = AutoModelForCausalLM.from_pretrained(
    READER_MODEL_NAME, quantization_config=bnb_config, device_map="auto"
)
tokenizer = AutoTokenizer.from_pretrained(READER_MODEL_NAME)

READER_LLM = pipeline(
    model=model,
    tokenizer=tokenizer,
    task="text-generation",
    do_sample=True,
    temperature=0.2,
    repetition_penalty=1.1,
    return_full_text=False,
    max_new_tokens=500,
)

config.json:   0%|          | 0.00/638 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/42.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/168 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'max_new_tokens', 'do_sample', 'temperature'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
# Chat Template Setup
prompt_in_chat_format = [
    {
        "role": "system",
        "content": """Using the information contained in the context, give a comprehensive answer to the question.
Respond only to the question asked, response should be concise and relevant to the question.
Provide the number of the source document when relevant.
If the answer cannot be deduced from the context, do not give an answer.""",
    },
    {
        "role": "user",
        "content": """Context:
{context}
---
Now here is the question you need to answer.

Question: {question}""",
    },
]

RAG_PROMPT_TEMPLATE = tokenizer.apply_chat_template(
    prompt_in_chat_format, tokenize=False, add_generation_prompt=True
)

In [ ]:
retrieved_docs_text = [
    doc.page_content for doc in retrieved_docs
]  # We only need the text of the documents
context = "\nExtracted documents:\n"
context += "".join(
    [f"Document {str(i)}:::\n" + doc for i, doc in enumerate(retrieved_docs_text)]
)

final_prompt = RAG_PROMPT_TEMPLATE.format(
    question="How to create a pipeline object?", context=context
)

# Redact an answer
answer = READER_LLM(final_prompt)[0]["generated_text"]
print(answer)

Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


The provided context does not include a question related to creating a pipeline object. Please provide a different context with a question related to creating a pipeline object for me to answer. Without further context, I am unable to provide an answer to this question.


## Reranking

In [ ]:
from rerankers import Reranker
RERANKER = Reranker(model_name="cross-encoder/ms-marco-MiniLM-L-6-v2")

Loading TransformerRanker model cross-encoder/ms-marco-MiniLM-L-6-v2 (this message can be suppressed by setting verbose=0)
No device set
Using device cuda
No dtype set
Using dtype torch.float32


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

...

Loaded model cross-encoder/ms-marco-MiniLM-L-6-v2
Using device cuda.
Using dtype torch.float32.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
from transformers import Pipeline
from typing import Tuple

def answer_with_rag(
    question: str,
    llm: pipeline, # Using the lower-case 'pipeline' type from transformers
    knowledge_index: FAISS,
    reranker: Optional[Reranker] = None,
    num_retrieved_docs: int = 30,
    num_docs_final: int = 5,
) -> Tuple[str, List[str]]:

    print("=> Retrieving documents...")
    relevant_docs = knowledge_index.similarity_search(
        query=question, k=num_retrieved_docs
    )
    # Extract page content
    relevant_docs_text = [doc.page_content for doc in relevant_docs]

    if reranker:
        print("===> Reranking documents...")
        # Rerank the strings
        rerank_results = reranker.rank(query=question, docs=relevant_docs_text)
        # Select top results
        final_docs = [res.document for res in rerank_results.results[:num_docs_final]]
    else:
        final_docs = relevant_docs_text[:num_docs_final]

    # Build the final prompt
    context = "\nExtracted documents:\n"
    context += "\n\n".join(
        [f"--- Document {i} ---\n{text}" for i, text in enumerate(final_docs)]
    )

    full_prompt = RAG_PROMPT_TEMPLATE.format(question=question, context=context)

    print("=> Generating answer...")
    answer = llm(full_prompt)[0]["generated_text"]

    return answer, final_docs

In [ ]:
# Run the query
question = "What is the meaning of 'Basic remuneration'?"

answer, relevant_docs = answer_with_rag(
    question, READER_LLM, KNOWLEDGE_VECTOR_DATABASE, reranker=RERANKER
)

print("\nFINAL ANSWER:\n")
print(answer)

=> Retrieving documents...
===> Reranking documents...


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=> Generating answer...

FINAL ANSWER:

The basic remuneration refers to the fixed amount of salary or wages that an employee receives as a base for their work, excluding any additional payments such as bonuses, commissions, or overtime pay. It is often calculated as a percentage of the employee's total earnings or based on industry standards and job responsibilities. The concept of basic remuneration is discussed in several documents provided, including Documents 1, 2, and 3, where provisions are made for its calculation, deposit, and payment.


In [ ]:
print("==================================Answer==================================")
print(f"{answer}")
print("==================================Source docs==================================")
for i, doc in enumerate(relevant_docs):
    print(f"Document {i}------------------------------------------------------------")
    print(doc)

==================================Answer==================================
The basic remuneration refers to the fixed amount of salary or wages that an employee receives as a base for their work, excluding any additional payments such as bonuses, commissions, or overtime pay. It is often calculated as a percentage of the employee's total earnings or based on industry standards and job responsibilities. The concept of basic remuneration is discussed in several documents provided, including Documents 1, 2, and 3, where provisions are made for its calculation, deposit, and payment.
==================================Source docs==================================
Document 0------------------------------------------------------------
Document(text='www.lawcommission.gov.np \n14 \n \nChapter-8 \nProvisions Relating to Remuneration \n34. Entitlement of labours to remuneration: (1) Each worker shall be entitled to \nreceive the remuneration and benefits from the date  on which he or she starts t

In [ ]:
import gc
torch.cuda.empty_cache()
gc.collect()

104

## Evaluate answers

In [ ]:
from tqdm.auto import tqdm
import pandas as pd
from typing import Optional, List, Tuple
import json

### Setting up agents for question generation

In [ ]:
import os
from huggingface_hub import InferenceClient
from openai import OpenAI
from google.colab import userdata

GROQ_API_KEY = userdata.get('GROQ_API_KEY')

client = OpenAI(
    api_key=GROQ_API_KEY,
    base_url="https://api.groq.com/openai/v1"
)

def call_llm(client_instance: OpenAI, prompt: str, system_prompt: str = None):
    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": prompt})

    response = client_instance.chat.completions.create(
        messages=messages,
        model="llama-3.3-70b-versatile",
        max_tokens=200,
        temperature=0,
        top_p=0.9,
    )
    return response.choices[0].message.content

In [ ]:
QA_generation_prompt = """
Your task is to write a factoid question and an answer given a context.
Your factoid question should be answerable with a specific, concise piece of factual information from the context.
Your factoid question should be formulated in the same style as questions users could ask in a search engine.
This means that your factoid question MUST NOT mention something like "according to the passage" or "context".

Provide your answer as follows:

Output:::
Factoid question: (your factoid question)
Answer: (your answer to the factoid question)

Now here is the context.

Context: {context}\n
Output:::"""

In [ ]:
import random

N_GENERATIONS = 10

print(f"Generating {N_GENERATIONS} QA couples...")

outputs = []
for sampled_context in tqdm(random.sample(docs_processed, N_GENERATIONS)):
    # Generate QA couple
    output_QA_couple = call_llm(
        client, QA_generation_prompt.format(context=sampled_context.page_content)
    )
    try:
        question = output_QA_couple.split("Factoid question: ")[-1].split("Answer: ")[0]
        answer = output_QA_couple.split("Answer: ")[-1]
        assert len(answer) < 300, "Answer is too long"
        outputs.append(
            {
                "context": sampled_context.page_content,
                "question": question,
                "answer": answer,
                "source_doc": sampled_context.metadata["source"],
            }
        )
    except Exception as e:
        ### ----------------> need to add something here as well to not let error pass
        print(f"Skipping due to error: {e}")
        continue

Generating 10 QA couples...


  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
outputs

[{'context': 'www.lawcommission.gov.np \n76 \n \n(3) Where a body corporate commits any offence punishab le under this \nAct, such body shall be punished with the fine, and if imprisonment is also \nimposable for such offence, the chief executive of such body shall be imprisoned \naccordingly. \n165. Appeal: (1) Any person who is not satisfied with any order o r decision made by \nthe Department or Office in accordance with this Ac t or the rules framed under \nthis Act may make an appeal to the Labour Court within thirty-five days. \n  (2) Any labour who is not satisfied with any decisio n made by the \nemployer to terminat e employment or with punishment imposed with respect t o \nmisconduct may make an appeal to the Labour Court within thirty-five days of the \ndate of receipt of a notice of that decision or punishment. \n(3) Notwithstanding anything contained in sub-section ( 2), nothing \nshall be deemed to bar the making of appeal accordi ngly if the bye-law of any \nenterprise p

### Setting up critique agents

In [ ]:
# @title
# question_groundedness_critique_prompt = """
# You will be given a context and a question.
# Your task is to provide a 'total rating' scoring how well one can answer the given question unambiguously with the given context.
# Give your answer on a scale of 1 to 5, where 1 means that the question is not answerable at all given the context, and 5 means that the question is clearly and unambiguously answerable with the context.

# Provide your answer as follows:

# Answer:::
# Evaluation: (your rationale for the rating, as a text)
# Total rating: (your rating, as a number between 1 and 5)

# You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

# Now here are the question and context.

# Question: {question}\n
# Context: {context}\n
# Answer::: """

In [ ]:
question_groundedness_critique_prompt = """
You are evaluating a question-answer dataset.

Given the context and question, rate how well the question is grounded in the context.

Evaluation criteria:
5 = answer clearly supported by context
4 = mostly supported
3 = partially supported
2 = weak support
1 = not supported

Respond ONLY in this format:

Evaluation: <short reasoning>
Total rating: <1-5>

Context:
{context}

Question:
{question}
"""

In [ ]:
# @title
# question_relevance_critique_prompt = """
# You will be given a question.
# Your task is to provide a 'total rating' representing how useful this question can be to machine learning developers building NLP applications with the Hugging Face ecosystem.
# Give your answer on a scale of 1 to 5, where 1 means that the question is not useful at all, and 5 means that the question is extremely useful.

# Provide your answer as follows:

# Answer:::
# Evaluation: (your rationale for the rating, as a text)
# Total rating: (your rating, as a number between 1 and 5)

# You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

# Now here is the question.

# Question: {question}\n
# Answer::: """

In [ ]:
question_relevance_critique_prompt = """
Evaluate whether the question is relevant to the given document collection.

A relevant question:
- relates to the information in the document
- could be answered using the document

5 = highly relevant
1 = completely irrelevant

Respond ONLY in this format:

Evaluation: <short reasoning>
Total rating: <1-5>

Question:
{question}
"""

In [ ]:
# @title
# question_standalone_critique_prompt = """
# You will be given a question.
# Your task is to provide a 'total rating' representing how context-independent this question is.
# Give your answer on a scale of 1 to 5, where 1 means that the question depends on additional information to be understood, and 5 means that the question makes sense by itself.
# For instance, if the question refers to a particular setting, like 'in the context' or 'in the document', the rating must be 1.
# The questions can contain obscure technical nouns or acronyms like Gradio, Hub, Hugging Face or Space and still be a 5: it must simply be clear to an operator with access to documentation what the question is about.

# For instance, "What is the name of the checkpoint from which the ViT model is imported?" should receive a 1, since there is an implicit mention of a context, thus the question is not independent from the context.

# Provide your answer as follows:

# Answer:::
# Evaluation: (your rationale for the rating, as a text)
# Total rating: (your rating, as a number between 1 and 5)

# You MUST provide values for 'Evaluation:' and 'Total rating:' in your answer.

# Now here is the question.

# Question: {question}\n
# Answer::: """

In [ ]:
question_standalone_critique_prompt = """
Evaluate whether the question is understandable without additional context.

5 = fully self-contained
1 = unclear without context

Respond ONLY in this format:

Evaluation: <short reasoning>
Total rating: <1-5>

Question:
{question}
"""

In [ ]:
print("Generating critique for each QA couple...")
for output in tqdm(outputs):
    evaluations = {
        "groundedness": call_llm(
            client,
            question_groundedness_critique_prompt.format(
                context=output["context"], question=output["question"]
            ),
        ),
        "relevance": call_llm(
            client,
            question_relevance_critique_prompt.format(question=output["question"]),
        ),
        "standalone": call_llm(
            client,
            question_standalone_critique_prompt.format(question=output["question"]),
        ),
    }
    try:
        for criterion, evaluation in evaluations.items():
            score, eval = (
                int(evaluation.split("Total rating: ")[-1].strip()),
                evaluation.split("Total rating: ")[-2].split("Evaluation: ")[1],
            )
            output.update(
                {
                    f"{criterion}_score": score,
                    f"{criterion}_eval": eval,
                }
            )
    except Exception as e:
        print(f"Skipping due to error: {e}")
        continue

Generating critique for each QA couple...


  0%|          | 0/10 [00:00<?, ?it/s]

In [ ]:
import pandas as pd
import datasets

pd.set_option("display.max_colwidth", None)

generated_questions = pd.DataFrame.from_dict(outputs)

print("Evaluation dataset before filtering:")
display(
    generated_questions[
        [
            "question",
            "answer",
            "groundedness_score",
            "relevance_score",
            "standalone_score",
        ]
    ]
)
generated_questions = generated_questions.loc[
    (generated_questions["groundedness_score"] >= 3)
    & (generated_questions["relevance_score"] >= 3)
    & (generated_questions["standalone_score"] >= 3)
]
print("============================================")
print("Final evaluation dataset:")
display(
    generated_questions[
        [
            "question",
            "answer",
            "groundedness_score",
            "relevance_score",
            "standalone_score",
        ]
    ]
)

eval_dataset = datasets.Dataset.from_pandas(
    generated_questions, split="train", preserve_index=False
)

Evaluation dataset before filtering:


,question,answer,groundedness_score,relevance_score,standalone_score
0,What is the time limit for making an appeal to the Labour Court?\n,thirty-five days,5,5,4
1,What is the condition for determining whether a work is of equal value for the purpose of remuneration?\n,"The nature of the concerned work, the time required for its performance, labour, skill and productivity.",5,5,4
2,What is required of an offender during the period of suspension of their imprisonment sentence\n,"to do a public work for free, or to assist in any work of the victim of the offence",5,5,4
3,What is required for making a collective bargaining or collective agreement?\n,Good faith,5,5,3
4,What is the maximum time period for paying compensation in installments?\n,one year,5,5,4
5,What is the maximum imprisonment term for failing to execute a Labour Court order?\n,One year,5,5,2
6,What is the time frame for taking a decision after commencing an action under sub-section (1) of the law?\n,Three months,5,5,2
7,When did the Interim Government of Nepal Act come into force?\n,"29th Chaitra, 2007",5,5,4
8,How many days does the Department have to settle an application made under sub-section (5)?\n,15 days,5,5,1
9,What happens to a labour supplier with a revoked license if they have financial liability towards the Government of Nepal or any labour?\n,"They shall not get something (the specific thing is not mentioned in the given context, but it implies they won't get a certain benefit or clearance).",5,5,4


Final evaluation dataset:


,question,answer,groundedness_score,relevance_score,standalone_score
0,What is the time limit for making an appeal to the Labour Court?\n,thirty-five days,5,5,4
1,What is the condition for determining whether a work is of equal value for the purpose of remuneration?\n,"The nature of the concerned work, the time required for its performance, labour, skill and productivity.",5,5,4
2,What is required of an offender during the period of suspension of their imprisonment sentence\n,"to do a public work for free, or to assist in any work of the victim of the offence",5,5,4
3,What is required for making a collective bargaining or collective agreement?\n,Good faith,5,5,3
4,What is the maximum time period for paying compensation in installments?\n,one year,5,5,4
7,When did the Interim Government of Nepal Act come into force?\n,"29th Chaitra, 2007",5,5,4
9,What happens to a labour supplier with a revoked license if they have financial liability towards the Government of Nepal or any labour?\n,"They shall not get something (the specific thing is not mentioned in the given context, but it implies they won't get a certain benefit or clearance).",5,5,4


## Benchmarking the RAG system

In [ ]:
from langchain_core.language_models import BaseChatModel
from langchain_core.vectorstores import VectorStore

def run_rag_tests(
    eval_dataset,
    llm,
    knowledge_index: VectorStore,
    output_file: str,
    reranker: Optional[Reranker] = None,
    verbose: Optional[bool] = True,
    test_settings: Optional[str] = None,  # To document the test settings used
):
    """Runs RAG tests on the given dataset and saves the results to the given output file."""
    try:  # load previous generations if they exist
        with open(output_file, "r") as f:
            outputs = json.load(f)
    except:
        outputs = []

    for example in tqdm(eval_dataset):
        question = example["question"]
        if question in [output["question"] for output in outputs]:
            continue

        answer, relevant_docs = answer_with_rag(
            question, llm, knowledge_index, reranker=reranker
        )
        if verbose:
            print("=======================================================")
            print(f"Question: {question}")
            print(f"Answer: {answer}")
            print(f'True answer: {example["answer"]}')
        result = {
            "question": question,
            "true_answer": example["answer"],
            "source_doc": example["source_doc"],
            "generated_answer": answer,
            # "retrieved_docs": [doc for doc in relevant_docs],
            "retrieved_docs": [
                doc.page_content if hasattr(doc, 'page_content') else str(doc)
                for doc in relevant_docs
            ]
        }
        ragas_result = {
            "question": question,
            "true_answer": example["answer"],      # ground truth
            "generated_answer": answer,            # model answer
            "retrieved_docs": [
                doc.page_content if hasattr(doc, 'page_content') else str(doc)
                for doc in relevant_docs
            ]
        }
        outputs.append(ragas_result)
        # if test_settings:
        #     result["test_settings"] = test_settings
        # outputs.append(result)

        # with open(output_file, "w") as f:
        #     json.dump(outputs, f)

    return outputs

In [ ]:
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores.utils import DistanceStrategy
from langchain_core.documents import Document
import os


def load_embeddings(
    langchain_docs: List[Document],
    chunk_size: int,
    embedding_model_name: Optional[str] = "thenlper/gte-small",
) -> FAISS:
    """
    Creates a FAISS index from the given embedding model and documents. Loads the index directly if it already exists.

    Args:
        langchain_docs: list of documents
        chunk_size: size of the chunks to split the documents into
        embedding_model_name: name of the embedding model to use

    Returns:
        FAISS index
    """
    # load embedding_model
    embedding_model = HuggingFaceEmbeddings(
        model_name=embedding_model_name,
        multi_process=True,
        model_kwargs={"device": "cuda"},
        encode_kwargs={
            "normalize_embeddings": True
        },  # set True to compute cosine similarity
    )

    # Check if embeddings already exist on disk
    index_name = (
        f"index_chunk:{chunk_size}_embeddings:{embedding_model_name.replace('/', '~')}"
    )
    index_folder_path = f"./data/indexes/{index_name}/"
    if os.path.isdir(index_folder_path):
        return FAISS.load_local(
            index_folder_path,
            embedding_model,
            distance_strategy=DistanceStrategy.COSINE,
            allow_dangerous_deserialization=True,
        )

    else:
        print("Index not found, generating it...")
        docs_processed = split_documents(
            chunk_size,
            langchain_docs,
            embedding_model_name,
        )
        knowledge_index = FAISS.from_documents(
            docs_processed, embedding_model, distance_strategy=DistanceStrategy.COSINE
        )
        knowledge_index.save_local(index_folder_path)
        return knowledge_index

### Evaluate the answer on the outputs

In [ ]:
!pip install ragas
!pip install langchain_huggingface

In [ ]:
# @title
# from langchain_huggingface import HuggingFaceEndpoint, HuggingFaceEmbeddings
# from ragas import evaluate
# from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
# from ragas.embeddings import LangchainEmbeddingsWrapper
# from ragas.llms import LangchainLLMWrapper
# from datasets import Dataset

# # llm = HuggingFaceEndpoint(
# #     repo_id="HuggingFaceH4/zephyr-7b-alpha",
# #     huggingfacehub_api_token=userdata.get('HF_TOKEN'),
# #     task="conversational",
# #     temperature=0.1,
# #     max_new_tokens=512,
# # )

# llm = HuggingFaceEndpoint(
#     repo_id="mistralai/Mistral-7B-Instruct-v0.3",
#     huggingfacehub_api_token=userdata.get("HF_TOKEN"),
#     task="text-generation",
#     temperature=0.1,
#     max_new_tokens=512,
# )

# ragas_llm = LangchainLLMWrapper(hf_llm)

# hf_embeddings = HuggingFaceEmbeddings(
#     model_name="thenlper/gte-small"
# )

# ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

# knowledge_index = load_embeddings(
#     RAW_KNOWLEDGE_BASE,
#     chunk_size=500,
#     embedding_model_name="thenlper/gte-small",
# )
# outputs = run_rag_tests(
#     eval_dataset=eval_dataset,
#     llm=READER_LLM,
#     knowledge_index=knowledge_index,
#     output_file=None, # has been changed to None for this instance
#     reranker=RERANKER,
#     verbose=False,
#     test_settings=None, # has been changed to None for this instance
# )

# ragas_dataset = [
#     {
#         "question": item["question"],
#         "ground_truth": item["true_answer"],
#         "answer": item["generated_answer"],
#         "contexts": item["retrieved_docs"]
#     }
#     for item in outputs
# ]

# print(f"The ragas_dataset: {ragas_dataset}")

# dataset = Dataset.from_list(ragas_dataset)

# score = evaluate(
#     dataset,
#     metrics=[
#         faithfulness,
#         answer_relevancy,
#         context_precision,
#         context_recall,
#     ],
#     llm=ragas_llm,
#     embeddings=ragas_embeddings,
# )
# df = score.to_pandas()

/tmp/ipykernel_3096/3489979816.py:3: DeprecationWarning: Importing faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import faithfulness
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_3096/3489979816.py:3: DeprecationWarning: Importing answer_relevancy from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import answer_relevancy
  from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall
/tmp/ipykernel_3096/3489979816.py:3: DeprecationWarning: Importing context_precision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import context_precision
  from ragas.metrics import faithfulne

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: thenlper/gte-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_3096/3489979816.py:30: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: thenlper/gte-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

=> Generating answer...
=> Retrieving documents...
===> Reranking documents...


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=> Generating answer...
The ragas_dataset: [{'question': 'What is the time limit for making an appeal to the Labour Court?\n', 'ground_truth': 'thirty-five days', 'answer': "The time limit for making an appeal to the Labour Court varies depending on the specific section cited in the extracts provided. Here's a breakdown:\n\n- In Extract 0, the time limit for appealing a decision made by the Labour Court is 35 days from the date of knowledge of the order or judgment. (Section 161)\n- In Extract 1, the time limit for applying to the Labour Court after being dismissed or punished for misconduct is 35 days from the date of receipt of the dismissal notice or punishment. (Section 162)\n- In Extract 3, the time limit for appealing a decision made by the Labour Court or an agreement reached through negotiation is 35 days from the date of receipt of the decision or agreement. (Section 166)\n- In Extract 4, the time limit for appealing a decision made by the Labour Court regarding the execution 

Evaluating:   0%|          | 0/28 [00:00<?, ?it/s]

ERROR:ragas.executor:Exception raised in Job[13]: ValueError(Model HuggingFaceH4/zephyr-7b-alpha is not supported for task text-generation and provider featherless-ai. Supported task: conversational.)
ERROR:ragas.executor:Exception raised in Job[8]: ValueError(Model HuggingFaceH4/zephyr-7b-alpha is not supported for task text-generation and provider featherless-ai. Supported task: conversational.)
ERROR:ragas.executor:Exception raised in Job[11]: ValueError(Model HuggingFaceH4/zephyr-7b-alpha is not supported for task text-generation and provider featherless-ai. Supported task: conversational.)
ERROR:ragas.executor:Exception raised in Job[5]: ValueError(Model HuggingFaceH4/zephyr-7b-alpha is not supported for task text-generation and provider featherless-ai. Supported task: conversational.)
ERROR:ragas.executor:Exception raised in Job[15]: ValueError(Model HuggingFaceH4/zephyr-7b-alpha is not supported for task text-generation and provider featherless-ai. Supported task: conversational

In [ ]:
from langchain_huggingface import HuggingFaceEndpoint, HuggingFaceEmbeddings

from ragas import evaluate
from ragas.metrics.collections import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.llms import LangchainLLMWrapper

from datasets import Dataset

# ------------------------
# evaluator LLM
# ------------------------
hf_llm = HuggingFaceEndpoint(
    repo_id="mistralai/Mistral-7B-Instruct-v0.3",
    huggingfacehub_api_token=userdata.get("HF_TOKEN"),
    task="text-generation",
    temperature=0.1,
    max_new_tokens=512,
)

ragas_llm = LangchainLLMWrapper(hf_llm)

# ------------------------
# evaluator embeddings
# ------------------------
hf_embeddings = HuggingFaceEmbeddings(
    model_name="thenlper/gte-small"
)

ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)

# ------------------------
# convert dataset
# ------------------------
dataset = Dataset.from_list(ragas_dataset)

# ------------------------
# evaluate
# ------------------------
score = evaluate(
    dataset,
    metrics=[
        faithfulness,
        answer_relevancy,
        context_precision,
        context_recall,
    ],
    llm=ragas_llm,
    embeddings=ragas_embeddings,
)

df = score.to_pandas()

print(df)

/tmp/ipykernel_3096/2971638276.py:27: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  ragas_llm = LangchainLLMWrapper(hf_llm)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: thenlper/gte-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/tmp/ipykernel_3096/2971638276.py:36: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  ragas_embeddings = LangchainEmbeddingsWrapper(hf_embeddings)


TypeError: All metrics must be initialised metric objects, e.g: metrics=[BleuScore(), AspectCritic()]

In [ ]:
EVALUATION_PROMPT = """###Task Description:
An instruction (might include an Input inside it), a response to evaluate, a reference answer that gets a score of 5, and a score rubric representing a evaluation criteria are given.
1. Write a detailed feedback that assess the quality of the response strictly based on the given score rubric, not evaluating in general.
2. After writing a feedback, write a score that is an integer between 1 and 5. You should refer to the score rubric.
3. The output format should look as follows: \"Feedback: {{write a feedback for criteria}} [RESULT] {{an integer number between 1 and 5}}\"
4. Please do not generate any other opening, closing, and explanations. Be sure to include [RESULT] in your output.

###The instruction to evaluate:
{instruction}

###Response to evaluate:
{response}

###Reference Answer (Score 5):
{reference_answer}

###Score Rubrics:
[Is the response correct, accurate, and factual based on the reference answer?]
Score 1: The response is completely incorrect, inaccurate, and/or not factual.
Score 2: The response is mostly incorrect, inaccurate, and/or not factual.
Score 3: The response is somewhat correct, accurate, and/or factual.
Score 4: The response is mostly correct, accurate, and factual.
Score 5: The response is completely correct, accurate, and factual.

###Feedback:"""

from langchain_core.prompts import (
    ChatPromptTemplate,
    HumanMessagePromptTemplate,
)
from langchain_core.messages import SystemMessage


evaluation_prompt_template = ChatPromptTemplate.from_messages(
    [
        SystemMessage(content="You are a fair evaluator language model."),
        HumanMessagePromptTemplate.from_template(EVALUATION_PROMPT),
    ]
)

In [ ]:
!pip install -U langchain-groq

In [ ]:
import os
from langchain_groq import ChatGroq

GROQ_API_KEY = userdata.get('GROQ_API_KEY')

# Initialize the groq model
eval_chat_model = ChatGroq(
    model="llama-3.3-70b-versatile",
    temperature=0,
    api_key=GROQ_API_KEY
)

evaluator_name = "OLLAMA_llama3"

In [ ]:
def evaluate_answers(
    answer_path: str,
    eval_chat_model,
    evaluator_name: str,
    evaluation_prompt_template: ChatPromptTemplate,
) -> None:
    """Evaluates generated answers. Modifies the given answer file in place for better checkpointing."""
    answers = []
    if os.path.isfile(answer_path):  # load previous generations if they exist
        answers = json.load(open(answer_path, "r"))

    for experiment in tqdm(answers):
        if f"eval_score_{evaluator_name}" in experiment:
            continue

        eval_prompt = evaluation_prompt_template.format_messages(
            instruction=experiment["question"],
            response=experiment["generated_answer"],
            reference_answer=experiment["true_answer"],
        )
        eval_result = eval_chat_model.invoke(eval_prompt)
        feedback, score = [
            item.strip() for item in eval_result.content.split("[RESULT]")
        ]
        experiment[f"eval_score_{evaluator_name}"] = score
        experiment[f"eval_feedback_{evaluator_name}"] = feedback

        with open(answer_path, "w") as f:
            json.dump(answers, f)

In [ ]:
if not os.path.exists("./output"):
    os.mkdir("./output")

for chunk_size in [200, 400, 500, 600]:  # Add other chunk sizes (in tokens) as needed
    for embeddings in ["thenlper/gte-small"]:  # Add other embeddings as needed
        for rerank in [True, False]:
            settings_name = f"chunk:{chunk_size}_embeddings:{embeddings.replace('/', '~')}_rerank:{rerank}_reader-model:{READER_MODEL_NAME.replace('/', '~')}"
            output_file_name = f"./output/rag_{settings_name}.json"

            print(f"Running evaluation for {settings_name}:")

            print("Loading knowledge base embeddings...")
            knowledge_index = load_embeddings(
                RAW_KNOWLEDGE_BASE,
                chunk_size=chunk_size,
                embedding_model_name=embeddings,
            )

            print("Running RAG...")

            run_rag_tests(
                eval_dataset=eval_dataset,
                llm=READER_LLM,
                knowledge_index=knowledge_index,
                output_file=output_file_name,
                reranker=RERANKER,
                verbose=False,
                test_settings=settings_name,
            )

            print("Running evaluation...")
            evaluate_answers(
                output_file_name,
                eval_chat_model,
                evaluator_name,
                evaluation_prompt_template,
            )

Running evaluation for chunk:200_embeddings:thenlper~gte-small_rerank:True_reader-model:HuggingFaceH4~zephyr-7b-beta:
Loading knowledge base embeddings...


ERROR:asyncio:Task exception was never retrieved
future: <Task finished name='Task-1135' coro=<as_completed.<locals>.sema_coro() done, defined at /usr/local/lib/python3.12/dist-packages/ragas/async_utils.py:75> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipykernel_2472/1490820368.py", line 47, in <cell line: 0>
    score = evaluate(
            ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ragas/_analytics.py", line 278, in wrapper
    result = func(*args, **kwargs)
             ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ragas/evaluation.py", line 484, in evaluate
    return run(_async_wrapper())
           ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/ragas/async_utils.py", line 156, in run
    return asyncio.run(coro)
           ^

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: thenlper/gte-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Running RAG...


  0%|          | 0/6 [00:00<?, ?it/s]

=> Retrieving documents...


Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


===> Reranking documents...
=> Generating answer...


In [ ]:
import glob

outputs = []
for file in glob.glob("./output/*.json"):
    output = pd.DataFrame(json.load(open(file, "r")))
    output["settings"] = file
    outputs.append(output)
result = pd.concat(outputs)

In [ ]:
# result["eval_score_OLLAMA_llama3"] = result["eval_score_OLLAMA_llama3"].apply(
#     lambda x: int(x) if isinstance(x, str) else 1
# )
# result["eval_score_OLLAMA_llama3"] = (result["eval_score_OLLAMA_llama3"] - 1) / 4

def safe_parse_score(x):
    try:
        return int(str(x).strip())
    except:
        return None

result["eval_score_OLLAMA_llama3"] = result["eval_score_OLLAMA_llama3"].apply(safe_parse_score)

In [ ]:
average_scores = result.groupby("settings")["eval_score_OLLAMA_llama3"].mean()
average_scores.sort_values()

In [ ]:
average_scores